> **ARCHIVAL ONLY** — This notebook is preserved for reference. The active pipeline has been refactored into the `turkey_audio_detection` Python package under `src/`. Use the CLI (`turkey-pipeline`) and Streamlit review app (`turkey-review`) for all new work. Do not run this notebook as part of standard workflow.

# Turkey Call Audio Detection Pipeline

**Objective:** Detect and classify turkey vocalizations (toms vs. hens) in WAV files collected from autonomous recording units (ARUs) at nesting sites in Rhode Island.

**Approach:** Use BirdNET to bootstrap candidate "Wild Turkey" labels, manually label clips as **Tom**, **Hen**, or **Background** via a Jupyter labeling widget, then train a PyTorch 3-class classifier on mel spectrogram features.

**Phases:**
1. Data Audit & Preprocessing
2. BirdNET Baseline Inference
3. Labeling Widget (Tom / Hen / Background)
4. Feature Extraction
5. PyTorch 3-Class Classifier Training
6. Full Inference & Output

In [ ]:
import struct, os, re, warnings
from pathlib import Path
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf
import librosa
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Project paths ──
PROJECT_DIR = Path(r"S:\Savanna Institute\Miscellaneous\Work\Dylan Bakner\Turkey Audio Detection")
DATA_DIR = PROJECT_DIR / "Data"
LABELS_CSV = PROJECT_DIR / "labels.csv"
DETECTIONS_CSV = PROJECT_DIR / "birdnet_detections.csv"
CLIPS_DIR = PROJECT_DIR / "clips"

print(f"Project directory: {PROJECT_DIR}")
print(f"Data directory:    {DATA_DIR}")
print(f"ARU folders:       {[d.name for d in sorted(DATA_DIR.iterdir()) if d.is_dir()]}")

## Phase 1: Data Audit & Preprocessing

In [ ]:
# ── 1a. Check WAV format from a sample file ──
sample_wav = next(DATA_DIR.rglob("*.wav"))
info = sf.info(str(sample_wav))
print(f"Sample file:   {sample_wav.name}")
print(f"Sample rate:   {info.samplerate} Hz")
print(f"Channels:      {info.channels}")
print(f"Duration:      {info.duration:.2f} sec ({info.duration/60:.2f} min)")
print(f"Format:        {info.format} / {info.subtype}")

SAMPLE_RATE = info.samplerate

In [ ]:
# ── 1b. Parse all WAV filenames into a DataFrame ──
# Filename format: <DeviceID>_<YYYYMMDD>_<HHMMSS>.wav

FILENAME_PATTERN = re.compile(r"^(\w+)_(\d{8})_(\d{6})\.wav$", re.IGNORECASE)

records = []
for aru_folder in sorted(DATA_DIR.iterdir()):
    if not aru_folder.is_dir():
        continue
    aru_id = aru_folder.name  # e.g., "ARU_01"
    for wav_path in sorted(aru_folder.rglob("*.wav")):
        match = FILENAME_PATTERN.match(wav_path.name)
        if not match:
            print(f"  WARNING: Skipping non-matching filename: {wav_path.name}")
            continue
        device_id = match.group(1)
        date_str = match.group(2)
        time_str = match.group(3)
        rec_datetime = datetime.strptime(f"{date_str}_{time_str}", "%Y%m%d_%H%M%S")
        records.append({
            "aru_id": aru_id,
            "device_id": device_id,
            "date": rec_datetime.date(),
            "time": rec_datetime.time(),
            "datetime": rec_datetime,
            "filepath": str(wav_path),
        })

df_files = pd.DataFrame(records)
print(f"Total WAV files found: {len(df_files)}")
df_files.head()

In [ ]:
# ── 1c. Filter pre-deployment test files ──
# Operational recordings are March 4–29, 2026. Anything before March 2026 is a test file.
from datetime import date as Date

DEPLOYMENT_START = Date(2026, 3, 1)

pre_deploy_mask = df_files["date"] < DEPLOYMENT_START
n_outliers = pre_deploy_mask.sum()
print(f"Pre-deployment test files to exclude: {n_outliers}")
if n_outliers > 0:
    print(df_files.loc[pre_deploy_mask, ["aru_id", "device_id", "date", "time", "filepath"]])

df_files = df_files[~pre_deploy_mask].reset_index(drop=True)
print(f"\nOperational WAV files: {len(df_files)}")

In [ ]:
# ── 1d. Compute sunrise times and tag files ──
from astral import LocationInfo
from astral.sun import sun

# Center of Rhode Island for all ARUs
RI_LOCATION = LocationInfo(
    name="Rhode Island",
    region="USA",
    timezone="US/Eastern",
    latitude=41.7,
    longitude=-71.5,
)

import zoneinfo
eastern = zoneinfo.ZoneInfo("US/Eastern")

def get_sunrise(date_obj):
    """Get sunrise time as a naive datetime in US/Eastern for a given date."""
    s = sun(RI_LOCATION.observer, date=date_obj, tzinfo=eastern)
    return s["sunrise"].replace(tzinfo=None)

# Compute sunrise and minutes-from-sunrise for each file
sunrises = {}
mins_from_sunrise = []
for _, row in df_files.iterrows():
    d = row["date"]
    if d not in sunrises:
        sunrises[d] = get_sunrise(d)
    sr = sunrises[d]
    rec_dt = datetime.combine(d, row["time"])
    delta = (rec_dt - sr).total_seconds() / 60.0
    mins_from_sunrise.append(delta)

df_files["sunrise"] = df_files["date"].map(lambda d: sunrises.get(d))
df_files["minutes_from_sunrise"] = mins_from_sunrise
df_files["in_prime_window"] = df_files["minutes_from_sunrise"].between(-90, 90)

print("Minutes-from-sunrise stats:")
print(df_files["minutes_from_sunrise"].describe().round(1))
print(f"\nFiles in prime window (-90 to +90 min): {df_files['in_prime_window'].sum()} / {len(df_files)}")

In [ ]:
# ── 1e. Summary table ──
summary = df_files.groupby("aru_id").agg(
    n_files=("filepath", "count"),
    date_min=("date", "min"),
    date_max=("date", "max"),
    n_prime=("in_prime_window", "sum"),
).reset_index()
summary["pct_prime"] = (summary["n_prime"] / summary["n_files"] * 100).round(1)
print(summary.to_string(index=False))

# Distribution plot: minutes from sunrise by ARU
fig, ax = plt.subplots(figsize=(10, 4))
for aru_id, grp in df_files.groupby("aru_id"):
    ax.hist(grp["minutes_from_sunrise"], bins=30, alpha=0.5, label=aru_id)
ax.axvline(0, color="orange", ls="--", lw=2, label="Sunrise")
ax.axvline(-90, color="red", ls=":", lw=1, label="Prime window")
ax.axvline(90, color="red", ls=":", lw=1)
ax.set_xlabel("Minutes from Sunrise")
ax.set_ylabel("Number of Files")
ax.set_title("Recording Times Relative to Sunrise")
ax.legend()
plt.tight_layout()
plt.show()

## Phase 2: BirdNET Baseline Inference

Run BirdNET on all WAV files and filter for "Wild Turkey" detections. This serves as both a baseline and a source of candidate positive clips for labeling.

In [ ]:
# ── 2a. Run BirdNET on prime-window WAV files only ──
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer

# analyzer = Analyzer()

# df_prime = df_files[df_files["in_prime_window"]].reset_index(drop=True)
# print(f"Running BirdNET on {len(df_prime)} prime-window files (out of {len(df_files)} total)")

# detections = []
# for idx, row in tqdm(df_prime.iterrows(), total=len(df_prime), desc="BirdNET inference"):
#     try:
#         recording = Recording(
#             analyzer,
#             row["filepath"],
#             lat=RI_LOCATION.latitude,
#             lon=RI_LOCATION.longitude,
#             date=row["datetime"],
#             min_conf=0.1,  # low threshold to capture candidates; we'll filter later
#         )
#         recording.analyze()
#         for det in recording.detections:
#             detections.append({
#                 "aru_id": row["aru_id"],
#                 "filepath": row["filepath"],
#                 "filename": Path(row["filepath"]).name,
#                 "date": row["date"],
#                 "rec_time": row["time"],
#                 "minutes_from_sunrise": row["minutes_from_sunrise"],
#                 "start_sec": det["start_time"],
#                 "end_sec": det["end_time"],
#                 "species": det["common_name"],
#                 "scientific_name": det["scientific_name"],
#                 "confidence": det["confidence"],
#             })
#     except Exception as e:
#         print(f"  ERROR on {row['filepath']}: {e}")

# df_detections = pd.DataFrame(detections)
# print(f"\nTotal BirdNET detections (all species, conf >= 0.1): {len(df_detections)}")

# # Save full detections
# df_detections.to_csv(DETECTIONS_CSV, index=False)
# print(f"Saved to {DETECTIONS_CSV}")

In [ ]:
# ── 2b. Filter for Wild Turkey and analyze confidence distribution ──
# (If re-running after a kernel restart, reload from CSV:)
df_detections = pd.read_csv(DETECTIONS_CSV)

df_turkey = df_detections[
    df_detections["species"].str.contains("Wild Turkey", case=False, na=False)
].copy()
print(f"Wild Turkey detections: {len(df_turkey)}")
print(f"  by ARU: {df_turkey.groupby('aru_id').size().to_dict()}")

# Confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_turkey["confidence"], bins=30, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("BirdNET Confidence")
axes[0].set_ylabel("Count")
axes[0].set_title("Wild Turkey Detection Confidence Distribution")

# Threshold sweep
thresholds = [0.1, 0.25, 0.5, 0.75, 0.9]
counts_at_threshold = [len(df_turkey[df_turkey["confidence"] >= t]) for t in thresholds]
axes[1].bar([str(t) for t in thresholds], counts_at_threshold, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Confidence Threshold")
axes[1].set_ylabel("Detections Retained")
axes[1].set_title("Detections at Various Confidence Thresholds")
for i, c in enumerate(counts_at_threshold):
    axes[1].text(i, c + 0.5, str(c), ha="center", fontsize=10)

plt.tight_layout()
plt.show()

# Top species detected overall
print("\nTop 15 species detected across all files:")
print(df_detections.groupby("species").size().sort_values(ascending=False).head(15).to_string())

## Phase 3: Labeling Widget

Extract candidate clips from BirdNET detections (positives) and random non-detection segments (negatives). Review each clip using an interactive widget with audio playback and spectrogram display. Label each clip as **Tom**, **Hen**, or **Background**.

In [ ]:
# ── 3a. Extract candidate clips (3-sec) ──
CLIP_DURATION = 3.0  # seconds

CLIPS_DIR.mkdir(exist_ok=True)

def extract_clip(wav_path, start_sec, clip_duration, out_path):
    """Extract a clip from a WAV file and save it."""
    info = sf.info(wav_path)
    start_frame = int(start_sec * info.samplerate)
    n_frames = int(clip_duration * info.samplerate)
    # Clamp to file length
    start_frame = max(0, min(start_frame, info.frames - n_frames))
    data, sr = sf.read(wav_path, start=start_frame, stop=start_frame + n_frames, dtype="float32")
    sf.write(str(out_path), data, sr)
    return sr

# ── Candidate positives: clips around BirdNET Wild Turkey detections ──
pos_clips = []
for idx, row in df_turkey.iterrows():
    # Center the 3-sec clip on the detection midpoint
    det_mid = (row["start_sec"] + row["end_sec"]) / 2
    clip_start = max(0, det_mid - CLIP_DURATION / 2)
    clip_name = f"pos_{row['aru_id']}_{row['filename'].replace('.wav', '')}_{clip_start:.1f}s.wav"
    clip_path = CLIPS_DIR / clip_name
    if not clip_path.exists():
        extract_clip(row["filepath"], clip_start, CLIP_DURATION, clip_path)
    pos_clips.append({
        "clip_path": str(clip_path),
        "source_file": row["filepath"],
        "clip_start_sec": clip_start,
        "candidate_type": "birdnet_positive",
        "birdnet_confidence": row["confidence"],
        "aru_id": row["aru_id"],
    })

print(f"Candidate positive clips extracted: {len(pos_clips)}")

# ── Candidate negatives: random 3-sec windows from files with NO turkey detection ──
turkey_files = set(df_turkey["filepath"].unique())
neg_candidates = df_files[~df_files["filepath"].isin(turkey_files)]

# Sample up to 2x the number of positive clips, stratified by ARU
np.random.seed(42)
n_neg_target = min(len(pos_clips) * 2, len(neg_candidates) * 3)  # up to 3 per negative file

neg_clips = []
for _, row in neg_candidates.iterrows():
    if len(neg_clips) >= n_neg_target:
        break
    try:
        finfo = sf.info(row["filepath"])
        max_start = finfo.duration - CLIP_DURATION
        if max_start <= 0:
            continue
        clip_start = np.random.uniform(0, max_start)
        clip_name = f"neg_{row['aru_id']}_{Path(row['filepath']).stem}_{clip_start:.1f}s.wav"
        clip_path = CLIPS_DIR / clip_name
        if not clip_path.exists():
            extract_clip(row["filepath"], clip_start, CLIP_DURATION, clip_path)
        neg_clips.append({
            "clip_path": str(clip_path),
            "source_file": row["filepath"],
            "clip_start_sec": clip_start,
            "candidate_type": "random_negative",
            "birdnet_confidence": None,
            "aru_id": row["aru_id"],
        })
    except Exception as e:
        print(f"  Error extracting negative from {row['filepath']}: {e}")

print(f"Candidate negative clips extracted: {len(neg_clips)}")

# Combine and shuffle
df_candidates = pd.DataFrame(pos_clips + neg_clips)
df_candidates = df_candidates.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Total candidate clips for review: {len(df_candidates)}")
print(f"  Positive candidates: {len(pos_clips)}")
print(f"  Negative candidates: {len(neg_clips)}")

In [ ]:
# ── 3b. Interactive labeling widget ──
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output

# Label mapping: 0 = Background, 1 = Hen, 2 = Tom, -1 = Skip
LABEL_MAP = {0: "Background", 1: "Hen", 2: "Tom", -1: "Skip"}

# Load existing labels if any
if LABELS_CSV.exists():
    df_labels = pd.read_csv(LABELS_CSV)
    labeled_paths = set(df_labels["clip_path"].values)
    print(f"Loaded {len(df_labels)} existing labels from {LABELS_CSV}")
else:
    df_labels = pd.DataFrame(columns=["clip_path", "label", "candidate_type", "aru_id", "notes"])
    labeled_paths = set()

# Filter to only unlabeled clips
unlabeled = df_candidates[~df_candidates["clip_path"].isin(labeled_paths)].reset_index(drop=True)
print(f"Clips remaining to label: {len(unlabeled)}")

# ── Widget state ──
current_idx = [0]

output = widgets.Output()
progress_label = widgets.Label()
info_label = widgets.HTML()

btn_tom = widgets.Button(description="Tom", button_style="success", layout=widgets.Layout(width="140px"))
btn_hen = widgets.Button(description="Hen", button_style="info", layout=widgets.Layout(width="140px"))
btn_not = widgets.Button(description="Other", button_style="danger", layout=widgets.Layout(width="140px"))
btn_skip = widgets.Button(description="Skip", button_style="warning", layout=widgets.Layout(width="100px"))
btn_prev = widgets.Button(description="◄ Prev", layout=widgets.Layout(width="80px"))
notes_input = widgets.Text(placeholder="Optional notes...", layout=widgets.Layout(width="300px"))

def save_label(label_value):
    """Save a label and advance to the next clip."""
    global df_labels
    idx = current_idx[0]
    if idx >= len(unlabeled):
        return
    row = unlabeled.iloc[idx]
    new_row = pd.DataFrame([{
        "clip_path": row["clip_path"],
        "label": label_value,
        "candidate_type": row["candidate_type"],
        "aru_id": row["aru_id"],
        "notes": notes_input.value,
    }])
    df_labels = pd.concat([df_labels, new_row], ignore_index=True)
    # Save incrementally
    df_labels.to_csv(LABELS_CSV, index=False)
    notes_input.value = ""
    current_idx[0] += 1
    show_clip()

def show_clip():
    """Display the current clip for review."""
    output.clear_output(wait=True)
    idx = current_idx[0]
    with output:
        if idx >= len(unlabeled):
            print(f"All {len(unlabeled)} clips have been labeled!")
            label_counts = df_labels["label"].value_counts()
            for lbl, name in LABEL_MAP.items():
                print(f"   {name}: {label_counts.get(lbl, 0)}")
            return
        row = unlabeled.iloc[idx]
        clip_path = row["clip_path"]

        # Load audio
        data, sr = sf.read(clip_path, dtype="float32")

        # Show info
        info_html = (
            f"<b>Clip {idx + 1} / {len(unlabeled)}</b> | "
            f"ARU: {row['aru_id']} | "
            f"Type: {row['candidate_type']} | "
            f"BirdNET conf: {row.get('birdnet_confidence', 'N/A')}"
        )
        info_label.value = info_html
        progress_label.value = f"Labeled: {len(df_labels)} | Remaining: {len(unlabeled) - idx}"

        # Audio player
        display(Audio(data=data, rate=sr, autoplay=False))

        # Mini spectrogram
        fig, ax = plt.subplots(figsize=(8, 2.5))
        S = librosa.feature.melspectrogram(y=data, sr=sr, n_mels=128, fmin=200, fmax=4000)
        S_dB = librosa.power_to_db(S, ref=np.max)
        librosa.display.specshow(S_dB, sr=sr, x_axis="time", y_axis="mel", ax=ax, fmin=200, fmax=4000)
        ax.set_title(Path(clip_path).stem, fontsize=9)
        ax.axhline(y=700, color="cyan", ls="--", lw=0.8, alpha=0.7)
        ax.axhline(y=1275, color="cyan", ls="--", lw=0.8, alpha=0.7)
        plt.tight_layout()
        plt.show()

btn_tom.on_click(lambda _: save_label(2))   # Tom = 2
btn_hen.on_click(lambda _: save_label(1))      # Hen = 1
btn_not.on_click(lambda _: save_label(0))        # Background = 0
btn_skip.on_click(lambda _: save_label(-1))      # Skip = -1
def go_prev(_):
    if current_idx[0] > 0:
        current_idx[0] -= 1
        # Remove last saved label
        global df_labels
        if len(df_labels) > 0:
            df_labels = df_labels.iloc[:-1]
            df_labels.to_csv(LABELS_CSV, index=False)
        show_clip()
btn_prev.on_click(go_prev)

# Layout
button_bar = widgets.HBox([btn_tom, btn_hen, btn_not, btn_skip, btn_prev, notes_input])
ui = widgets.VBox([info_label, progress_label, output, button_bar])
display(ui)

# Show first clip
show_clip()

## Phase 4: Feature Extraction

Extract mel spectrogram features from each labeled clip using librosa. Compute per-mel-bin statistics (mean, std, max, min, skew) to create a fixed-size feature vector per clip. Cache to disk.

In [ ]:
# ── 4a. Extract mel spectrogram features for labeled clips ──
import h5py
from scipy.stats import skew as scipy_skew

EMBEDDINGS_FILE = PROJECT_DIR / "features.h5"

# Label mapping: 0 = Background, 1 = Hen, 2 = Tom
LABEL_MAP = {0: "Background", 1: "Hen", 2: "Tom"}

# Feature extraction parameters
TARGET_SR = 48000
N_MELS = 128
FMIN = 200
FMAX = 6000
N_FFT = 2048
HOP_LENGTH = 512

def extract_mel_features(wav_path, sr=TARGET_SR):
    """Extract mel spectrogram statistics as a fixed-size feature vector."""
    data, file_sr = sf.read(wav_path, dtype="float32")
    if data.ndim > 1:
        data = data.mean(axis=1)  # mono
    if file_sr != sr:
        data = librosa.resample(data, orig_sr=file_sr, target_sr=sr)
    S = librosa.feature.melspectrogram(
        y=data, sr=sr, n_mels=N_MELS, n_fft=N_FFT,
        hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX,
    )
    S_dB = librosa.power_to_db(S, ref=np.max)
    # Per-mel-bin statistics across time: mean, std, max, min, skew → 5 × 128 = 640 features
    feat = np.concatenate([
        S_dB.mean(axis=1),
        S_dB.std(axis=1),
        S_dB.max(axis=1),
        S_dB.min(axis=1),
        scipy_skew(S_dB, axis=1),
    ])
    return feat.astype(np.float32)

# Reload labels
df_labels = pd.read_csv(LABELS_CSV)
# Drop skipped (label == -1); keep 0, 1, 2
df_labeled = df_labels[df_labels["label"].isin([0, 1, 2])].reset_index(drop=True)
print(f"Labeled clips (excluding skips): {len(df_labeled)}")
for lbl, name in LABEL_MAP.items():
    print(f"  {name} ({lbl}): {(df_labeled['label'] == lbl).sum()}")

# Extract for all labeled clips
feat_list = []
valid_indices = []
for idx, row in tqdm(df_labeled.iterrows(), total=len(df_labeled), desc="Extracting features"):
    try:
        feat = extract_mel_features(row["clip_path"])
        feat_list.append(feat)
        valid_indices.append(idx)
    except Exception as e:
        print(f"  Error on {row['clip_path']}: {e}")

X = np.array(feat_list, dtype=np.float32)
y = df_labeled.loc[valid_indices, "label"].values.astype(np.int64)
aru_ids = df_labeled.loc[valid_indices, "aru_id"].values

print(f"\nFeatures extracted: {X.shape[0]} clips × {X.shape[1]} dims")
for lbl, name in LABEL_MAP.items():
    print(f"  {name}: {int((y == lbl).sum())}")

# Save to HDF5
with h5py.File(EMBEDDINGS_FILE, "w") as f:
    f.create_dataset("X", data=X)
    f.create_dataset("y", data=y)
    f.create_dataset("aru_ids", data=aru_ids.astype("S"))
    f.create_dataset("clip_paths", data=np.array([row for row in df_labeled.loc[valid_indices, "clip_path"].values], dtype="S"))
print(f"Saved to {EMBEDDINGS_FILE}")

## Phase 5: PyTorch 3-Class Classifier Training

Train a 3-class classifier (MLP) on mel spectrogram features using leave-one-ARU-out cross-validation. Classes: **Background (0)**, **Hen (1)**, **Tom (2)**.

In [ ]:
# ── 5a. Load features and prepare data ──
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix

# Label mapping: 0 = Background, 1 = Hen, 2 = Tom
LABEL_MAP = {0: "Background", 1: "Hen", 2: "Tom"}
NUM_CLASSES = 3
FEATURE_DIM = N_MELS * 5  # 128 mel bins × 5 statistics = 640

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load cached features
with h5py.File(EMBEDDINGS_FILE, "r") as f:
    X = f["X"][:]
    y = f["y"][:]
    aru_ids = np.array([s.decode() for s in f["aru_ids"][:]])

unique_arus = np.unique(aru_ids)
print(f"Loaded {X.shape[0]} clips, {X.shape[1]} feature dims")
print(f"ARUs: {unique_arus}")
print("Class balance:")
for lbl, name in LABEL_MAP.items():
    print(f"  {name}: {int((y == lbl).sum())}")


class EmbeddingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class TurkeyCallClassifier(nn.Module):
    """3-class classifier: Background (0), Hen (1), Tom (2)."""
    def __init__(self, input_dim=640, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# ── 5b. Leave-one-ARU-out cross-validation ──

def train_one_fold(X_train, y_train, X_val, y_val, n_epochs=100, lr=1e-3, patience=10):
    """Train the MLP on one fold and return val metrics + trained model."""
    train_ds = EmbeddingDataset(X_train, y_train)
    val_ds = EmbeddingDataset(X_val, y_val)

    # Weighted sampler for class imbalance (inverse frequency per class)
    class_counts = np.bincount(y_train.astype(int), minlength=NUM_CLASSES).astype(float)
    class_counts[class_counts == 0] = 1.0  # avoid division by zero
    sample_weights = 1.0 / class_counts[y_train.astype(int)]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=64, sampler=sampler)
    val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

    model = TurkeyCallClassifier(input_dim=X_train.shape[1], num_classes=NUM_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss()

    best_f1 = 0
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    patience_counter = 0

    for epoch in range(n_epochs):
        # Train
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # Validate
        model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                logits = model(X_batch)
                all_logits.append(logits.cpu())
                all_labels.append(y_batch)
        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels).numpy()
        preds = all_logits.argmax(dim=1).numpy()
        f1 = f1_score(all_labels, preds, average="macro", zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Final eval with best model
    model.load_state_dict(best_state)
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch.to(device))
            all_logits.append(logits.cpu())
            all_labels.append(y_batch)
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels).numpy()
    probs = torch.softmax(all_logits, dim=1).numpy()
    preds = probs.argmax(axis=1)

    metrics = {
        "precision_macro": precision_score(all_labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(all_labels, preds, average="macro", zero_division=0),
        "f1_macro": f1_score(all_labels, preds, average="macro", zero_division=0),
        "f1_tom": f1_score(all_labels == 2, preds == 2, zero_division=0),
        "f1_hen": f1_score(all_labels == 1, preds == 1, zero_division=0),
        "n_val": len(all_labels),
        "n_val_tom": int((all_labels == 2).sum()),
        "n_val_hen": int((all_labels == 1).sum()),
        "n_val_not_turkey": int((all_labels == 0).sum()),
    }
    return model, metrics, all_labels, preds

# ── Run leave-one-ARU-out CV ──
results = []
best_models = {}
all_cv_labels = []
all_cv_preds = []

for held_out_aru in unique_arus:
    train_mask = aru_ids != held_out_aru
    val_mask = aru_ids == held_out_aru

    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val = X[val_mask], y[val_mask]

    if len(np.unique(y_val)) < 2:
        print(f"  Fold {held_out_aru}: skipping — only one class in val set")
        continue

    n_per_class = {LABEL_MAP[lbl]: int((y_val == lbl).sum()) for lbl in range(NUM_CLASSES)}
    print(f"Fold: held out {held_out_aru} | train={len(y_train)} | val={len(y_val)} {n_per_class}")
    model, metrics, fold_labels, fold_preds = train_one_fold(X_train, y_train, X_val, y_val)
    metrics["held_out_aru"] = held_out_aru
    results.append(metrics)
    best_models[held_out_aru] = model
    all_cv_labels.extend(fold_labels)
    all_cv_preds.extend(fold_preds)
    print(f"  → F1(macro)={metrics['f1_macro']:.3f}  F1(tom)={metrics['f1_tom']:.3f}  F1(hen)={metrics['f1_hen']:.3f}")

df_results = pd.DataFrame(results)
print("\n── Cross-Validation Summary ──")
print(df_results.to_string(index=False))
print(f"\nMean F1 (macro): {df_results['f1_macro'].mean():.3f} ± {df_results['f1_macro'].std():.3f}")
print(f"Mean F1 (tom): {df_results['f1_tom'].mean():.3f}")
print(f"Mean F1 (hen): {df_results['f1_hen'].mean():.3f}")

# Pooled confusion matrix
all_cv_labels = np.array(all_cv_labels)
all_cv_preds = np.array(all_cv_preds)
class_names = [LABEL_MAP[i] for i in range(NUM_CLASSES)]
print("\n── Pooled Classification Report ──")
print(classification_report(all_cv_labels, all_cv_preds, target_names=class_names, zero_division=0))

cm = confusion_matrix(all_cv_labels, all_cv_preds, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues", ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Leave-One-ARU-Out Confusion Matrix (pooled)")
plt.tight_layout()
plt.show()

In [ ]:
# ── 5c. Train final model on ALL labeled data and save ──
MODEL_PATH = PROJECT_DIR / "turkey_classifier.pt"

print("Training final model on all labeled data...")
final_model = TurkeyCallClassifier(input_dim=X.shape[1], num_classes=NUM_CLASSES).to(device)

train_ds = EmbeddingDataset(X, y)
class_counts = np.bincount(y.astype(int), minlength=NUM_CLASSES).astype(float)
class_counts[class_counts == 0] = 1.0
sample_weights = 1.0 / class_counts[y.astype(int)]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
train_loader = DataLoader(train_ds, batch_size=64, sampler=sampler)

optimizer = torch.optim.AdamW(final_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
criterion = nn.CrossEntropyLoss()

for epoch in range(100):
    final_model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(final_model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
    scheduler.step()

torch.save(final_model.state_dict(), MODEL_PATH)
print(f"Final model saved to {MODEL_PATH}")

## Phase 6: Full-Dataset Inference & Output

Run the trained 3-class model on all WAV files. Classify each window as Background, Hen, or Tom. Merge overlapping detections into events. Output event-level timestamps (with call type) and per-day activity summaries.

In [ ]:
# ── 6a. Run inference on all WAV files ──
EVENTS_CSV = PROJECT_DIR / "turkey_call_events.csv"
SUMMARY_CSV = PROJECT_DIR / "activity_summary.csv"

WINDOW_SEC = 3.0
STRIDE_SEC = 1.5
CONF_THRESHOLD = 0.5  # minimum probability to keep a turkey detection

# Label mapping
LABEL_MAP = {0: "Background", 1: "Hen", 2: "Tom"}

# Load final model
final_model = TurkeyCallClassifier(input_dim=FEATURE_DIM, num_classes=NUM_CLASSES).to(device)
final_model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
final_model.eval()

def extract_mel_features_from_array(data, sr):
    """Extract mel spectrogram features from a numpy audio array."""
    S = librosa.feature.melspectrogram(
        y=data, sr=sr, n_mels=N_MELS, n_fft=N_FFT,
        hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX,
    )
    S_dB = librosa.power_to_db(S, ref=np.max)
    feat = np.concatenate([
        S_dB.mean(axis=1),
        S_dB.std(axis=1),
        S_dB.max(axis=1),
        S_dB.min(axis=1),
        scipy_skew(S_dB, axis=1),
    ])
    return feat.astype(np.float32)

all_events = []

for _, row in tqdm(df_files.iterrows(), total=len(df_files), desc="Full inference"):
    filepath = row["filepath"]
    try:
        data, file_sr = sf.read(filepath, dtype="float32")
    except Exception as e:
        print(f"  Error reading {filepath}: {e}")
        continue

    if data.ndim > 1:
        data = data.mean(axis=1)
    if file_sr != TARGET_SR:
        data = librosa.resample(data, orig_sr=file_sr, target_sr=TARGET_SR)

    duration = len(data) / TARGET_SR
    n_samples_window = int(WINDOW_SEC * TARGET_SR)
    n_samples_stride = int(STRIDE_SEC * TARGET_SR)

    # Slide windows across the file
    features = []
    window_starts = []
    pos = 0
    while pos + n_samples_window <= len(data):
        window = data[pos : pos + n_samples_window]
        feat = extract_mel_features_from_array(window, TARGET_SR)
        features.append(feat)
        window_starts.append(pos / TARGET_SR)
        pos += n_samples_stride

    if len(features) == 0:
        continue

    # Run classifier on all windows at once
    X_file = np.array(features, dtype=np.float32)
    with torch.no_grad():
        logits = final_model(torch.tensor(X_file).to(device)).cpu()
    probs = torch.softmax(logits, dim=1).numpy()

    # Keep windows where a turkey class (1=Hen or 2=Tom) exceeds threshold
    for win_idx in range(len(probs)):
        pred_class = probs[win_idx].argmax()
        pred_prob = probs[win_idx, pred_class]
        if pred_class >= 1 and pred_prob >= CONF_THRESHOLD:
            start_sec = window_starts[win_idx]
            end_sec = start_sec + WINDOW_SEC
            all_events.append({
                "aru_id": row["aru_id"],
                "date": row["date"],
                "rec_time": row["time"],
                "filepath": filepath,
                "start_sec": start_sec,
                "end_sec": end_sec,
                "call_type": LABEL_MAP[pred_class],
                "confidence": float(pred_prob),
                "prob_background": float(probs[win_idx, 0]),
                "prob_hen": float(probs[win_idx, 1]),
                "prob_tom": float(probs[win_idx, 2]),
                "minutes_from_sunrise": row["minutes_from_sunrise"],
            })

df_events = pd.DataFrame(all_events)
print(f"\nRaw detection windows: {len(df_events)}")
if len(df_events) > 0:
    print(f"  Tom: {(df_events['call_type'] == 'Tom').sum()}")
    print(f"  Hen: {(df_events['call_type'] == 'Hen').sum()}")

In [ ]:
# ── 6b. Merge overlapping windows into discrete events (NMS), per call type ──

def merge_events(df, max_gap_sec=1.0):
    """Merge adjacent/overlapping detection windows into discrete events."""
    if len(df) == 0:
        return df
    merged = []
    for (aru, date, filepath, call_type), grp in df.groupby(["aru_id", "date", "filepath", "call_type"]):
        grp = grp.sort_values("start_sec").reset_index(drop=True)
        cur_start = grp.iloc[0]["start_sec"]
        cur_end = grp.iloc[0]["end_sec"]
        cur_conf = [grp.iloc[0]["confidence"]]
        cur_mfs = grp.iloc[0]["minutes_from_sunrise"]
        cur_rec_time = grp.iloc[0]["rec_time"]

        for i in range(1, len(grp)):
            row = grp.iloc[i]
            if row["start_sec"] <= cur_end + max_gap_sec:
                cur_end = max(cur_end, row["end_sec"])
                cur_conf.append(row["confidence"])
            else:
                merged.append({
                    "aru_id": aru,
                    "date": date,
                    "rec_time": cur_rec_time,
                    "filepath": filepath,
                    "call_type": call_type,
                    "start_sec": cur_start,
                    "end_sec": cur_end,
                    "confidence": float(np.mean(cur_conf)),
                    "max_confidence": float(np.max(cur_conf)),
                    "minutes_from_sunrise": cur_mfs,
                })
                cur_start = row["start_sec"]
                cur_end = row["end_sec"]
                cur_conf = [row["confidence"]]
                cur_mfs = row["minutes_from_sunrise"]
                cur_rec_time = row["rec_time"]

        merged.append({
            "aru_id": aru,
            "date": date,
            "rec_time": cur_rec_time,
            "filepath": filepath,
            "call_type": call_type,
            "start_sec": cur_start,
            "end_sec": cur_end,
            "confidence": float(np.mean(cur_conf)),
            "max_confidence": float(np.max(cur_conf)),
            "minutes_from_sunrise": cur_mfs,
        })
    return pd.DataFrame(merged)

df_merged = merge_events(df_events)
print(f"Merged events: {len(df_merged)} (from {len(df_events)} raw windows)")
if len(df_merged) > 0:
    print(f"  Tom: {(df_merged['call_type'] == 'Tom').sum()}")
    print(f"  Hen: {(df_merged['call_type'] == 'Hen').sum()}")

# Compute absolute event time (rec_time + start_sec offset)
if len(df_merged) > 0:
    df_merged["event_time"] = pd.to_datetime(df_merged["date"].astype(str) + " " + df_merged["rec_time"].astype(str)) + \
        pd.to_timedelta(df_merged["start_sec"], unit="s")

# Save
df_merged.to_csv(EVENTS_CSV, index=False)
print(f"Saved event log to {EVENTS_CSV}")
df_merged.head(10)

In [ ]:
# ── 6c. Activity summary and plots ──

# Summary: turkey calls per day per ARU, broken out by call type
df_summary = df_merged.groupby(["aru_id", "date", "call_type"]).agg(
    n_calls=("start_sec", "count"),
    mean_confidence=("confidence", "mean"),
).reset_index()

# Also build a pivot: one row per (aru, date) with tom and hen counts
df_pivot = df_summary.pivot_table(
    index=["aru_id", "date"], columns="call_type", values="n_calls", fill_value=0
).reset_index()
df_pivot.columns.name = None
# Ensure both columns exist
for col in ["Tom", "Hen"]:
    if col not in df_pivot.columns:
        df_pivot[col] = 0

# Merge sunrise info
sunrise_map = df_files[["date", "sunrise"]].drop_duplicates().set_index("date")["sunrise"]
df_pivot["sunrise"] = df_pivot["date"].map(sunrise_map)

df_summary.to_csv(SUMMARY_CSV, index=False)
print(f"Activity summary saved to {SUMMARY_CSV}")
print(f"\n── Turkey Calls by ARU and Type ──")
type_summary = df_merged.groupby(["aru_id", "call_type"]).size().unstack(fill_value=0)
print(type_summary.to_string())

# ── Plot 1: Daily activity by ARU (stacked: tom vs hen) ──
fig, axes = plt.subplots(len(unique_arus), 1, figsize=(12, 3 * len(unique_arus)), sharex=True)
if len(unique_arus) == 1:
    axes = [axes]

for ax, aru in zip(axes, sorted(unique_arus)):
    grp = df_pivot[df_pivot["aru_id"] == aru]
    dates = pd.to_datetime(grp["date"])
    ax.bar(dates, grp["Tom"], alpha=0.8, label="Tom", color="steelblue")
    ax.bar(dates, grp["Hen"], bottom=grp["Tom"], alpha=0.8, label="Hen", color="salmon")
    ax.set_ylabel("Detections")
    ax.set_title(f"{aru} — Daily Turkey Call Detections")
    ax.legend()
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

# ── Plot 2: Activity vs. minutes from sunrise, by call type ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for call_type, ax, color in [("Tom", axes[0], "steelblue"), ("Hen", axes[1], "salmon")]:
    grp = df_merged[df_merged["call_type"] == call_type]
    for aru in sorted(unique_arus):
        aru_grp = grp[grp["aru_id"] == aru]
        ax.hist(aru_grp["minutes_from_sunrise"], bins=30, alpha=0.5, label=aru)
    ax.axvline(0, color="orange", ls="--", lw=2, label="Sunrise")
    ax.set_xlabel("Minutes from Sunrise")
    ax.set_ylabel("Number of Events")
    ax.set_title(f"{call_type} Events Relative to Sunrise")
    ax.legend()
plt.tight_layout()
plt.show()

print(f"\n── Summary Statistics ──")
print(f"Total turkey call events detected: {len(df_merged)}")
if len(df_merged) > 0:
    print(f"  Tom: {(df_merged['call_type'] == 'Tom').sum()}")
    print(f"  Hen: {(df_merged['call_type'] == 'Hen').sum()}")
    print(f"Date range: {df_merged['date'].min()} to {df_merged['date'].max()}")
    print(f"Events in prime window (-90 to +90 min): {df_merged['minutes_from_sunrise'].between(-90, 90).sum()}")
    print(f"Events outside prime window: {(~df_merged['minutes_from_sunrise'].between(-90, 90)).sum()}")